# Filtro IIR pasa-bajas para una señal IQ con GNU Radio

Este cuaderno genera una señal IQ sintética, aplica un filtro IIR recursivo de un polo a I y Q por separado, y comprueba que la salida conserva la forma y queda más suave.

`filter.single_pole_iir_filter_ff` procesa muestras `float`, mientras que IQ es complejo. Por eso el flujo es: `vector_source_c` → `complex_to_float` → dos filtros IIR (I y Q) → `float_to_complex` → `vector_sink_c`.

Para ejecutar como script, exporte esta celda de código a `script.py` y ejecute `python3 script.py`.

In [ ]:
import numpy as np
from gnuradio import blocks, filter, gr


class IIRFilterFlowgraph(gr.top_block):
    def __init__(self, iq_samples: np.ndarray, alpha: float = 0.1):
        super().__init__("IIR IQ low-pass filter")
        if not 0.0 < alpha <= 1.0:
            raise ValueError("alpha must be in (0, 1]")

        self.input_samples = np.asarray(iq_samples, dtype=np.complex64).reshape(-1)
        self.source = blocks.vector_source_c(self.input_samples.tolist(), False)
        self.to_float = blocks.complex_to_float(1)
        self.iir_i = filter.single_pole_iir_filter_ff(alpha, 1)
        self.iir_q = filter.single_pole_iir_filter_ff(alpha, 1)
        self.to_complex = blocks.float_to_complex(1)
        self.sink = blocks.vector_sink_c()

        self.connect(self.source, self.to_float)
        self.connect((self.to_float, 0), self.iir_i, (self.to_complex, 0))
        self.connect((self.to_float, 1), self.iir_q, (self.to_complex, 1))
        self.connect(self.to_complex, self.sink)


def smoothness(samples: np.ndarray) -> float:
    """Mean squared change between adjacent complex samples."""
    return float(np.mean(np.abs(np.diff(samples)) ** 2))


In [ ]:
# Synthetic IQ: a low-frequency complex tone plus high-frequency noise.
sample_count = 5_000
seed = 42
alpha = 0.1
rng = np.random.default_rng(seed)
n = np.arange(sample_count, dtype=np.float32)
tone = np.exp(1j * 2 * np.pi * 0.02 * n)
noise = 0.8 * (rng.standard_normal(sample_count) + 1j * rng.standard_normal(sample_count))
iq_input = (tone + noise).astype(np.complex64)

flowgraph = IIRFilterFlowgraph(iq_input, alpha=alpha)
flowgraph.run()
iq_output = np.asarray(flowgraph.sink.data(), dtype=np.complex64)

input_power = float(np.mean(np.abs(iq_input) ** 2))
output_power = float(np.mean(np.abs(iq_output) ** 2))
input_smoothness = smoothness(iq_input)
output_smoothness = smoothness(iq_output)

print(f"Input shape:  {iq_input.shape}")
print(f"Output shape: {iq_output.shape}")
print(f"Input power:   {input_power:.4f}")
print(f"Output power:  {output_power:.4f}")
print(f"Input variation:  {input_smoothness:.4f}")
print(f"Output variation: {output_smoothness:.4f}")

# Simple flowgraph test: same number of samples, valid complex data, and smoothing.
assert iq_output.shape == iq_input.shape, "The filter changed the sample count"
assert np.isfinite(iq_output).all(), "The filter produced non-finite samples"
assert output_smoothness < input_smoothness, "The IIR filter did not smooth the signal"
print("PASS: GNU Radio IIR filter produced a same-shape, smoother IQ signal.")